# NeuroScan AI - GPU training on Google Colab

Trains all four models with the **same code as the repository** (`train.py`, `evaluate.py`), so checkpoints are drop-in compatible with the web app.

The training recipe fixes the over/under-fitting of the original models: duplicate-aware validation split, brain cropping, stronger augmentation, LP-FT with discriminative learning rates, label smoothing, EMA, early stopping on validation loss and temperature calibration. See the README for the evidence.

**Runtime → Change runtime type → GPU (T4 or better)**, then run the cells in order.

In [ ]:
import torch
print("PyTorch", torch.__version__, "| CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "Select a GPU runtime first (Runtime > Change runtime type)."

## 1 · Get the code

In [ ]:
REPO_URL = "https://github.com/ashwin8332/Neuroscan.git"  # change to your fork if needed
BRANCH = "main"
import os
if not os.path.exists("/content/neuroscan"):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/neuroscan
%cd /content/neuroscan
!pip install -q -r requirements.txt

## 2 · Get the dataset

**Option A - Kaggle:** create an API token (Kaggle → Settings → API → *Create New Token*) and upload `kaggle.json` when prompted.

**Option B - Google Drive:** set `DRIVE_ZIP` to a zip of the dataset in your Drive.

In [ ]:
import pathlib, shutil, zipfile

DATA_DIR = pathlib.Path("/content/brain-tumor-mri-dataset")
DRIVE_ZIP = None  # e.g. "/content/drive/MyDrive/brain-tumor-mri-dataset.zip"

def locate(root):
    for p in [root, *root.rglob("*")]:
        if p.is_dir() and (p / "Training").is_dir() and (p / "Testing").is_dir():
            return p
    return None

if not (DATA_DIR / "Training").is_dir():
    staging = pathlib.Path("/content/_dataset")
    staging.mkdir(exist_ok=True)
    if DRIVE_ZIP:
        from google.colab import drive
        drive.mount("/content/drive")
        zipfile.ZipFile(DRIVE_ZIP).extractall(staging)
    else:
        from google.colab import files
        print("Upload kaggle.json")
        files.upload()
        os.makedirs("/root/.kaggle", exist_ok=True)
        shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
        os.chmod("/root/.kaggle/kaggle.json", 0o600)
        !pip install -q kaggle
        !kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p /content/_dataset --unzip
    found = locate(staging)
    assert found, "Could not find a folder containing Training/ and Testing/"
    shutil.move(str(found), str(DATA_DIR))

for split in ("Training", "Testing"):
    print(split, {d.name: len(list(d.iterdir())) for d in sorted((DATA_DIR / split).iterdir()) if d.is_dir()})

## 3 · Audit the data

Counts near-duplicate images inside `Training/` (they leak into a random validation split) and test images that have a near-copy in `Training/` (they make test accuracy optimistic).

In [ ]:
!python scripts/audit_dataset.py --data-dir {DATA_DIR}

## 4 · Train

Roughly 10-25 minutes per CNN on a T4 and longer for ViT-B/16. Every run writes `models/<name>.pth` and `results/<name>_history.json`.

In [ ]:
!python train.py --model all --data-dir {DATA_DIR}

## 5 · Evaluate and plot

In [ ]:
# Standard test set (comparable with published numbers) ...
!python evaluate.py --data-dir {DATA_DIR} --tta
# ... and the leakage-free subset: test images with no near-copy in Training/
!python evaluate.py --data-dir {DATA_DIR} --tta --exclude-leaked --results-dir results/leakage_free
!python compare.py

## 6 · Download checkpoints and results

In [ ]:
!zip -qr neuroscan_trained.zip models results
from google.colab import files
files.download("neuroscan_trained.zip")

### After downloading

1. Unzip and copy `models/*.pth` into the project's `models/` folder and `results/*` into `results/`.
2. `best_vit_model.pth` (~330 MB) is git-ignored because it exceeds GitHub's 100 MB limit; keep it locally or host it as a release asset.
3. Start the app with `python server.py` and open http://127.0.0.1:8000. The *Training health* tab shows the new curves.